In [0]:
# ================================================================
# NOTEBOOK: nb_gold_returns_analysis
#
# PURPOSE:
#   Return, refund and customer-return analysis.
#
# RUN:
#   Daily, after Silver notebooks complete.
#
# READS:
#   silver/returns
#   silver/orders
#   silver/orderitems
#   silver/products
#   silver/customers
#
# WRITES:
#   gold/returns_analysis/
#
# GRAIN:
#   1 row per ReturnID
#
# ================================================================
# IMPORTANT DATA-MODEL LIMITATION
# ================================================================
#
# The Returns source contains OrderID but does not contain a reliable
# OrderItemID or ProductID for historical return records.
#
# For multi-item orders, the exact returned product cannot be
# identified.
#
# To avoid joining one return with every item in the order, this
# notebook selects the highest-value item from each order as a
# representative proxy.
#
# Exact metrics:
#   - Return count
#   - Refund amount
#   - Return reason
#   - Return status
#   - Customer analysis
#   - Days from order to return
#   - Refund processing time
#
# Approximate metrics:
#   - Proxy product
#   - Proxy category
#   - Proxy brand
#   - Proxy seller
#   - Estimated margin impact
#
# Do not describe ProxyProductID as the actual returned product.
#
# Production recommendation:
#   Capture OrderItemID directly in the Returns source when the
#   return transaction is created.
# ================================================================


import builtins

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ================================================================
# CONFIGURATION
# ================================================================

STORAGE = (
    "abfss://source@stshopsensedevhj.dfs.core.windows.net"
)

GOLD_PATH = (
    f"{STORAGE}/gold/returns_analysis/"
)


# ================================================================
# HELPER: FILTER SOFT-DELETED RECORDS WHEN COLUMN EXISTS
# ================================================================

def keep_active_records(df):
    """
    Keeps active records when the DataFrame contains _is_deleted.

    Null _is_deleted values are treated as False.
    """

    if "_is_deleted" in df.columns:
        return df.filter(
            F.coalesce(
                F.col("_is_deleted"),
                F.lit(False)
            ) == False
        )

    return df


# ================================================================
# STEP 1: READ SILVER TABLES
# ================================================================

returns = (
    spark.read
    .format("delta")
    .load(f"{STORAGE}/silver/returns/")
)

orders = (
    spark.read
    .format("delta")
    .load(f"{STORAGE}/silver/orders/")
)

items = (
    spark.read
    .format("delta")
    .load(f"{STORAGE}/silver/orderitems/")
)

products = (
    spark.read
    .format("delta")
    .load(f"{STORAGE}/silver/products/")
)

customers = (
    spark.read
    .format("delta")
    .load(f"{STORAGE}/silver/customers/")
)


# Keep active records only
returns = keep_active_records(returns)
orders = keep_active_records(orders)
items = keep_active_records(items)
products = keep_active_records(products)
customers = keep_active_records(customers)


# Keep only current customer SCD Type 2 record
if "is_current" in customers.columns:
    customers = customers.filter(
        F.col("is_current") == True
    )


print("\n[READ COUNTS]")

print(
    f"Returns  : {returns.count()}"
)

print(
    f"Orders   : {orders.count()}"
)

print(
    f"Items    : {items.count()}"
)

print(
    f"Products : {products.count()}"
)

print(
    f"Customers: {customers.count()}"
)


# ================================================================
# STEP 2: PREPARE ORDER CONTEXT
#
# SellerID is renamed to OrderSellerID to prevent conflict with
# Product SellerID.
# ================================================================

order_context = (
    orders
    .select(
        "OrderID",

        F.col("CustomerID")
        .alias("OrderCustomerID"),

        "OrderDate",
        "TotalAmount",
        "PaymentMethod",
        "IsPrimeOrder",
        "ShippingCity",
        "ShippingState",

        F.col("SellerID")
        .alias("OrderSellerID")
    )
    .dropDuplicates(["OrderID"])
)


# Join returns with exact order information
returns_with_order = (
    returns
    .join(
        order_context,
        on="OrderID",
        how="left"
    )
)


# Returns may already contain CustomerID.
# Use Returns.CustomerID first and Orders.CustomerID as fallback.
if "CustomerID" in returns.columns:

    returns_with_order = (
        returns_with_order
        .withColumn(
            "FinalCustomerID",
            F.coalesce(
                F.col("CustomerID"),
                F.col("OrderCustomerID")
            )
        )
    )

else:

    returns_with_order = (
        returns_with_order
        .withColumn(
            "FinalCustomerID",
            F.col("OrderCustomerID")
        )
    )


# ================================================================
# STEP 3: SELECT ONE PROXY PRODUCT PER ORDER
#
# Rule:
#   Select the item with the highest TotalPrice.
#
# Tie-breaker:
#   Select the lowest OrderItemID so the result remains deterministic.
#
# This does not identify the actual returned item.
# ================================================================

proxy_window = (
    Window
    .partitionBy("OrderID")
    .orderBy(
        F.col("TotalPrice").desc_nulls_last(),
        F.col("OrderItemID").asc_nulls_last()
    )
)


proxy_item_per_order = (
    items

    .withColumn(
        "_proxy_rank",
        F.row_number().over(proxy_window)
    )

    .filter(
        F.col("_proxy_rank") == 1
    )

    .select(
        "OrderID",

        F.col("OrderItemID")
        .alias("ProxyOrderItemID"),

        F.col("ProductID")
        .alias("ProxyProductID"),

        F.col("Quantity")
        .alias("ProxyPurchasedQuantity"),

        F.col("TotalPrice")
        .alias("ProxyItemValue")
    )
)


# ================================================================
# STEP 4: PREPARE PROXY PRODUCT CONTEXT
# ================================================================

proxy_product_context = (
    products

    .select(
        F.col("ProductID")
        .alias("ProxyProductID"),

        F.col("ProductName")
        .alias("ProxyProductName"),

        F.col("Category")
        .alias("ProxyCategory"),

        F.col("SubCategory")
        .alias("ProxySubCategory"),

        F.col("Brand")
        .alias("ProxyBrand"),

        F.col("SellerID")
        .alias("ProxySellerID"),

        F.col("ListPrice")
        .alias("ProxyListPrice"),

        F.col("GrossMarginPct")
        .alias("ProxyGrossMarginPct")
    )

    .dropDuplicates(["ProxyProductID"])
)


# ================================================================
# STEP 5: PREPARE CUSTOMER CONTEXT
# ================================================================

customer_context = (
    customers

    .select(
        F.col("CustomerID")
        .alias("FinalCustomerID"),

        "FullName",
        "City",
        "State",
        "Segment",
        "IsPrimeBool"
    )

    .dropDuplicates(["FinalCustomerID"])
)


# ================================================================
# STEP 6: JOIN ALL CONTEXT
# ================================================================

returns_with_context = (
    returns_with_order

    .join(
        proxy_item_per_order,
        on="OrderID",
        how="left"
    )

    .join(
        proxy_product_context,
        on="ProxyProductID",
        how="left"
    )

    .join(
        customer_context,
        on="FinalCustomerID",
        how="left"
    )
)


# ================================================================
# STEP 7: ADD BUSINESS-DERIVED COLUMNS
# ================================================================

gold_df = (
    returns_with_context

    # ------------------------------------------------------------
    # RETURN TIMING
    # ------------------------------------------------------------

    .withColumn(
        "DaysFromOrderToReturn",
        F.datediff(
            F.col("ReturnDate"),
            F.col("OrderDate")
        )
    )

    .withColumn(
        "IsQuickReturn",
        F.coalesce(
            F.col("DaysFromOrderToReturn") <= 1,
            F.lit(False)
        )
    )

    .withColumn(
        "IsLateReturn",
        F.coalesce(
            F.col("DaysFromOrderToReturn") > 7,
            F.lit(False)
        )
    )

    # ------------------------------------------------------------
    # ORDER-LEVEL REFUND ANALYSIS
    #
    # This compares RefundAmount with complete order TotalAmount.
    # It is not an exact product refund percentage.
    # ------------------------------------------------------------

    .withColumn(
        "OrderRefundPct",

        F.when(
            F.col("RefundAmount").isNotNull() &
            F.col("TotalAmount").isNotNull() &
            (F.col("TotalAmount") > 0),

            F.round(
                (
                    F.col("RefundAmount") /
                    F.col("TotalAmount")
                ) * 100,
                2
            )
        )

        .otherwise(
            F.lit(0.0)
        )
    )

    .withColumn(
        "OrderRefundBand",

        F.when(
            F.col("OrderRefundPct") >= 95,
            "NEAR_FULL_ORDER_VALUE"
        )

        .when(
            F.col("OrderRefundPct") >= 50,
            "HIGH_ORDER_VALUE"
        )

        .when(
            F.col("OrderRefundPct") > 0,
            "LOW_ORDER_VALUE"
        )

        .otherwise(
            "NO_REFUND"
        )
    )

    # ------------------------------------------------------------
    # ESTIMATED MARGIN IMPACT
    #
    # This uses the proxy product's gross margin percentage.
    # Therefore, it is an estimate.
    # ------------------------------------------------------------

    .withColumn(
        "EstimatedMarginLost",

        F.when(
            F.col("RefundAmount").isNotNull() &
            F.col("ProxyGrossMarginPct").isNotNull(),

            F.round(
                F.col("RefundAmount") *
                F.col("ProxyGrossMarginPct") /
                100,
                2
            )
        )

        .otherwise(
            F.lit(0.0)
        )
    )

    # ------------------------------------------------------------
    # RETURN-REASON CLASSIFICATION
    #
    # These are reason-based flags.
    # They do not prove legal or financial responsibility.
    # ------------------------------------------------------------

    .withColumn(
        "IsSellerFaultReason",

        F.coalesce(
            F.col("ReturnReason").isin(
                "DEFECTIVE_PRODUCT",
                "WRONG_ITEM",
                "DAMAGED_PRODUCT",
                "DAMAGED_IN_TRANSIT"
            ),
            F.lit(False)
        )
    )

    .withColumn(
        "IsCustomerFaultReason",

        F.coalesce(
            F.col("ReturnReason").isin(
                "CHANGED_MIND",
                "SIZE_MISMATCH"
            ),
            F.lit(False)
        )
    )

    # ------------------------------------------------------------
    # POTENTIAL RISK INDICATOR
    #
    # This is a rule-based flag, not proof of fraudulent behaviour.
    # ------------------------------------------------------------

    .withColumn(
        "IsPotentiallySuspicious",

        F.coalesce(
            (
                (F.col("IsQuickReturn") == True) &

                (F.col("OrderRefundPct") >= 95) &

                (
                    F.upper(
                        F.trim(
                            F.col("ConditionOnReturn")
                        )
                    ) == "SEALED"
                ) &

                (
                    ~F.col("ReturnReason").isin(
                        "DEFECTIVE_PRODUCT",
                        "WRONG_ITEM",
                        "DAMAGED_PRODUCT",
                        "DAMAGED_IN_TRANSIT"
                    )
                )
            ),
            F.lit(False)
        )
    )

    # ------------------------------------------------------------
    # PROXY-MAPPING INFORMATION
    # ------------------------------------------------------------

    .withColumn(
        "ProductMappingMethod",
        F.lit("HIGHEST_VALUE_ITEM_PROXY")
    )

    .withColumn(
        "IsProductMappingExact",
        F.lit(False)
    )

    .withColumn(
        "HasProxyProduct",
        F.col("ProxyProductID").isNotNull()
    )

    # ------------------------------------------------------------
    # TIME DIMENSIONS
    # ------------------------------------------------------------

    .withColumn(
        "ReturnYear",
        F.year("ReturnDate")
    )

    .withColumn(
        "ReturnMonth",
        F.month("ReturnDate")
    )

    .withColumn(
        "ReturnDayOfWeek",
        F.dayofweek("ReturnDate")
    )

    .withColumn(
        "IsWeekendReturn",
        F.coalesce(
            F.col("ReturnDayOfWeek").isin(1, 7),
            F.lit(False)
        )
    )

    # ------------------------------------------------------------
    # GOLD METADATA
    # ------------------------------------------------------------

    .withColumn(
        "_gold_load_ts",
        F.current_timestamp()
    )
)


# ================================================================
# STEP 8: VALIDATE GOLD GRAIN
#
# The proxy joins must not duplicate ReturnID records.
# ================================================================

source_row_count = returns.count()

gold_row_count = gold_df.count()


source_distinct_returns = (
    returns
    .select("ReturnID")
    .distinct()
    .count()
)


gold_distinct_returns = (
    gold_df
    .select("ReturnID")
    .distinct()
    .count()
)


print("\n[GRAIN VALIDATION]")

print(
    f"Source return rows       : {source_row_count}"
)

print(
    f"Gold rows                : {gold_row_count}"
)

print(
    f"Source distinct ReturnID : {source_distinct_returns}"
)

print(
    f"Gold distinct ReturnID   : {gold_distinct_returns}"
)


assert source_row_count == gold_row_count, (
    "Gold row count changed after joins. "
    "A join may have duplicated return records."
)


assert source_distinct_returns == gold_distinct_returns, (
    "Gold contains duplicated or missing ReturnID values."
)


print(
    "[PASS] Gold maintains one row per return."
)


# ================================================================
# STEP 9: PROXY-COVERAGE VALIDATION
# ================================================================

print("\n[PROXY PRODUCT COVERAGE]")


(
    gold_df

    .agg(
        F.count("*")
        .alias("TotalReturns"),

        F.count("ProxyProductID")
        .alias("ReturnsWithProxyProduct"),

        F.count(
            F.when(
                F.col("ProxyProductID").isNull(),
                1
            )
        )
        .alias("ReturnsWithoutProxyProduct")
    )

    .show(
        truncate=False
    )
)


# ================================================================
# STEP 10: WRITE GOLD
# ================================================================

(
    gold_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )

    .partitionBy(
        "ReturnYear",
        "ReturnMonth"
    )

    .save(
        GOLD_PATH
    )
)


print(
    f"\n[DONE] Gold Returns written: "
    f"{gold_row_count} rows"
)


# ================================================================
# STEP 11: BUSINESS SUMMARY
# ================================================================

total = gold_row_count


refunded = (
    gold_df
    .filter(
        F.col("ReturnStatus") == "REFUNDED"
    )
    .count()
)


rejected = (
    gold_df
    .filter(
        F.col("ReturnStatus") == "REJECTED"
    )
    .count()
)


seller_fault_reason = (
    gold_df
    .filter(
        F.col("IsSellerFaultReason") == True
    )
    .count()
)


customer_fault_reason = (
    gold_df
    .filter(
        F.col("IsCustomerFaultReason") == True
    )
    .count()
)


potentially_suspicious = (
    gold_df
    .filter(
        F.col("IsPotentiallySuspicious") == True
    )
    .count()
)


financial_summary = (
    gold_df

    .agg(
        F.coalesce(
            F.sum("RefundAmount"),
            F.lit(0.0)
        ).alias("TotalRefundAmount"),

        F.coalesce(
            F.sum("EstimatedMarginLost"),
            F.lit(0.0)
        ).alias("TotalEstimatedMarginLost")
    )

    .first()
)


total_refund = (
    financial_summary["TotalRefundAmount"]
)


total_estimated_margin = (
    financial_summary["TotalEstimatedMarginLost"]
)


def percentage(numerator, denominator):
    """
    Safely calculates a Python percentage.
    """

    if denominator == 0:
        return 0.0

    return builtins.round(
        numerator / denominator * 100,
        1
    )


print("\n[SUMMARY]")


print(
    f"Approved / Refunded: "
    f"{refunded} "
    f"({percentage(refunded, total)}%)"
)


print(
    f"Rejected: "
    f"{rejected} "
    f"({percentage(rejected, total)}%)"
)


print(
    f"Seller-fault reasons: "
    f"{seller_fault_reason} "
    f"({percentage(seller_fault_reason, total)}%)"
)


print(
    f"Customer-fault reasons: "
    f"{customer_fault_reason} "
    f"({percentage(customer_fault_reason, total)}%)"
)


print(
    f"Potentially suspicious: "
    f"{potentially_suspicious}"
)


print(
    f"Total refund value: "
    f"₹{total_refund:,.2f}"
)


print(
    f"Estimated margin lost: "
    f"₹{total_estimated_margin:,.2f}"
)


# ================================================================
# STEP 12: RETURN-REASON BREAKDOWN
#
# This analysis is exact because it uses Returns-table columns.
# ================================================================

print("\n[RETURN REASON BREAKDOWN]")


(
    gold_df

    .groupBy(
        "ReasonCategory",
        "ReturnReason"
    )

    .agg(
        F.countDistinct("ReturnID")
        .alias("ReturnCount"),

        F.round(
            F.avg("RefundAmount"),
            2
        )
        .alias("AvgRefundAmount"),

        F.round(
            F.avg("DaysFromOrderToReturn"),
            1
        )
        .alias("AvgDaysToReturn")
    )

    .orderBy(
        F.col("ReturnCount").desc()
    )

    .show(
        15,
        truncate=False
    )
)


# ================================================================
# STEP 13: RETURN SUMMARY BY PROXY CATEGORY
#
# Do not call this "category return rate".
#
# Meaning:
#   Returns associated with orders whose highest-value item belonged
#   to the displayed category.
# ================================================================

print("\n[RETURN SUMMARY BY PROXY CATEGORY]")


(
    gold_df

    .groupBy(
        "ProxyCategory"
    )

    .agg(
        F.countDistinct("ReturnID")
        .alias("AssociatedReturnCount"),

        F.round(
            F.sum("RefundAmount"),
            2
        )
        .alias("TotalRefundAmount"),

        F.round(
            F.avg("OrderRefundPct"),
            2
        )
        .alias("AvgOrderRefundPct"),

        F.round(
            F.sum("EstimatedMarginLost"),
            2
        )
        .alias("EstimatedMarginLost"),

        F.count(
            F.when(
                F.col("IsSellerFaultReason") == True,
                1
            )
        )
        .alias("SellerFaultReasonCount")
    )

    .orderBy(
        F.col("AssociatedReturnCount").desc()
    )

    .show(
        truncate=False
    )
)


# ================================================================
# STEP 14: REFUND-PROCESSING TIME
# ================================================================

print("\n[REFUND PROCESSING TIME]")


(
    gold_df

    .filter(
        F.col("DaysToProcess").isNotNull()
    )

    .agg(
        F.round(
            F.avg("DaysToProcess"),
            1
        )
        .alias("AvgDays"),

        F.percentile_approx(
            "DaysToProcess",
            0.5
        )
        .alias("MedianDays"),

        F.max("DaysToProcess")
        .alias("MaxDays"),

        F.min("DaysToProcess")
        .alias("MinDays"),

        F.count(
            F.when(
                F.col("DaysToProcess") > 7,
                1
            )
        )
        .alias("SlowProcessed_7PlusDays")
    )

    .show(
        truncate=False
    )
)


# ================================================================
# STEP 15: DISPLAY SAMPLE RETURNS
# ================================================================

display(
    gold_df

    .select(
        "ReturnID",
        "OrderID",
        "FinalCustomerID",
        "FullName",

        "ProxyProductID",
        "ProxyProductName",
        "ProxyCategory",
        "ProxySellerID",

        "ProductMappingMethod",
        "IsProductMappingExact",

        "ReturnReason",
        "ReasonCategory",
        "ReturnStatus",

        "RefundAmount",
        "OrderRefundPct",
        "OrderRefundBand",

        "DaysFromOrderToReturn",
        "DaysToProcess",

        "IsSellerFaultReason",
        "IsCustomerFaultReason",
        "IsPotentiallySuspicious",

        "EstimatedMarginLost",
        "ReturnDate"
    )

    .orderBy(
        F.col("EstimatedMarginLost").desc()
    )

    .limit(15)
)


[READ COUNTS]
Returns  : 409
Orders   : 3015
Items    : 5945
Products : 300
Customers: 500

[GRAIN VALIDATION]
Source return rows       : 409
Gold rows                : 409
Source distinct ReturnID : 409
Gold distinct ReturnID   : 409
[PASS] Gold maintains one row per return.

[PROXY PRODUCT COVERAGE]
+------------+-----------------------+--------------------------+
|TotalReturns|ReturnsWithProxyProduct|ReturnsWithoutProxyProduct|
+------------+-----------------------+--------------------------+
|409         |409                    |0                         |
+------------+-----------------------+--------------------------+


[DONE] Gold Returns written: 409 rows

[SUMMARY]
Approved / Refunded: 2 (0.5%)
Rejected: 89 (21.8%)
Seller-fault reasons: 104 (25.4%)
Customer-fault reasons: 123 (30.1%)
Potentially suspicious: 0
Total refund value: ₹7,836,797.80
Estimated margin lost: ₹2,820,903.12

[RETURN REASON BREAKDOWN]
+-------------------+-----------------+-----------+---------------+---

ReturnID,OrderID,FinalCustomerID,FullName,ProxyProductID,ProxyProductName,ProxyCategory,ProxySellerID,ProductMappingMethod,IsProductMappingExact,ReturnReason,ReasonCategory,ReturnStatus,RefundAmount,OrderRefundPct,OrderRefundBand,DaysFromOrderToReturn,DaysToProcess,IsSellerFaultReason,IsCustomerFaultReason,IsPotentiallySuspicious,EstimatedMarginLost,ReturnDate
RET000004,ORD0000385,CUST00362,Arjun Verma,PROD0212,Nike Beauty Product 212,Beauty,SELL043,HIGHEST_VALUE_ITEM_PROXY,false,CHANGED_MIND,CUSTOMER_PREFERENCE,PROCESSING,46927.83,99.43,NEAR_FULL_ORDER_VALUE,11,null,false,true,false,22975.87,2024-05-16T21:48:50.000Z
RET000009,ORD0002906,CUST00192,Amit Kumar,PROD0299,Adidas Sports Product 299,Sports,SELL013,HIGHEST_VALUE_ITEM_PROXY,false,SIZE_MISMATCH,OTHER,PROCESSING,47252.11,99.76,NEAR_FULL_ORDER_VALUE,8,null,false,true,false,22487.28,2024-01-12T16:35:04.000Z
RET000146,ORD0002845,CUST00431,Kavya Reddy,PROD0006,Prestige Electronics Product 6,Electronics,SELL017,HIGHEST_VALUE_ITEM_PROXY,false,QUALITY_ISSUE,PRODUCT_ISSUE,PROCESSING,44440.36,95.31,NEAR_FULL_ORDER_VALUE,12,null,false,false,false,21886.88,2024-01-26T06:03:01.000Z
RET000244,ORD0000189,CUST00151,Anita Joshi,PROD0069,Lakme Clothing Product 69,Clothing,SELL042,HIGHEST_VALUE_ITEM_PROXY,false,SIZE_MISMATCH,FIT_ISSUE,REJECTED,47855.41,98.07,NEAR_FULL_ORDER_VALUE,12,6,false,true,false,21755.07,2024-05-14T11:11:03.000Z
RET000063,ORD0001738,CUST00163,Kavya Gupta,PROD0084,Adidas Clothing Product 84,Clothing,SELL050,HIGHEST_VALUE_ITEM_PROXY,false,CHANGED_MIND,CUSTOMER_PREFERENCE,PROCESSING,43202.65,99.49,NEAR_FULL_ORDER_VALUE,4,null,false,true,false,20404.61,2024-03-22T05:43:44.000Z
RET000262,ORD0001422,CUST00274,Priya Nair,PROD0263,Samsung Sports Product 263,Sports,SELL029,HIGHEST_VALUE_ITEM_PROXY,false,WRONG_ITEM,FULFILLMENT_ISSUE,PROCESSING,43530.70,93.91,HIGH_ORDER_VALUE,9,null,true,false,false,20045.89,2024-03-20T02:33:50.000Z
RET000274,ORD0001252,CUST00076,Vikram Singh,PROD0225,Bosch Beauty Product 225,Beauty,SELL012,HIGHEST_VALUE_ITEM_PROXY,false,NOT_AS_DESCRIBED,PRODUCT_ISSUE,PROCESSING,40774.48,85.61,HIGH_ORDER_VALUE,11,null,false,false,false,19571.75,2024-05-18T13:53:38.000Z
RET000172,ORD0002033,CUST00282,Amit Joshi,PROD0212,Nike Beauty Product 212,Beauty,SELL043,HIGHEST_VALUE_ITEM_PROXY,false,DEFECTIVE_PRODUCT,PRODUCT_ISSUE,PROCESSING,39962.78,90.83,HIGH_ORDER_VALUE,6,null,true,false,false,19565.78,2024-01-09T17:36:09.000Z
RET000325,ORD0000363,CUST00228,Sneha Gupta,PROD0250,Adidas Beauty Product 250,Beauty,SELL013,HIGHEST_VALUE_ITEM_PROXY,false,NOT_AS_DESCRIBED,WRONG_PRODUCT,REJECTED,41367.03,91.82,HIGH_ORDER_VALUE,13,5,false,false,false,18470.38,2024-01-14T06:17:12.000Z
RET000290,ORD0002005,CUST00259,Pooja Rao,PROD0188,Penguin Homekitchen Product 188,Homekitchen,SELL044,HIGHEST_VALUE_ITEM_PROXY,false,CHANGED_MIND,CUSTOMER_PREFERENCE,PROCESSING,37182.73,84.93,HIGH_ORDER_VALUE,16,null,false,true,false,18193.51,2024-02-11T09:52:13.000Z
